In [ ]:
!pip install Keras-Preprocessing

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 15.8 MB/s eta 0:00:00


In [ ]:
!pip install np_utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for np_utils: filename=np_utils-0.6.0-py3-none-any.whl size=56437 sha256=0e118931b5eeeda1bc353b422f7d585091edd213322fb289462bd00028c65896
  Stored in directory: /root/.cache/pip/wheels/b6/c7/50/2307607f44366dd021209f660045f8d51cb976514d30be7cc7
Successfully built np_utils


In [ ]:
from keras.models import load_model
from tf_data import TF_Data
import argparse
import pandas
import os

In [ ]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
# pre-processing
def preprrocess_text(df):
  i = 0
  # lower case
  df["normalized_headline"] = df["Title"].str.lower()

  # remove stop word
  for text in df['normalized_headline']:
    stop_words = set(stopwords.words("english"))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    text = " ".join(filtered_words)

    df.loc[i,'normalized_headline'] = text
    print(text)
    i += 1

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
filename = './data/Combine_Label_ActualValue.csv'

df = pandas.read_csv(filename)
#headlines = df['normalized_headline']

# pre-processing
preprrocess_text(df)

rally legs, broad reach, --- gains s&p 500 show advances across board, large tech companies
rally legs, broad reach, --- gains s&p 500 show advances across board, large tech companies
u.s. stock futures slip u.k. polls suggest conservatives fail reach majority; s&p 500 futures fall 0.1%, british pound slides ahead final election results
stocks fall; tech shares take brunt; tech stocks s&p 500's worst-performing sector second straight session still 18% year
s&p 500 slips fed raises interest rates; investors sold technology shares late session, dragging nasdaq composite
equities: s&p 500 edges --- index retreats fed raises rates, treasurys rally dollar pulls back
u.s. stocks climb tech sector bounces back; dow, s&p 500 hit fresh records; nasdaq composite climbs 1.4%
equities: dow, s&p 500 hit records tech bounces
u.s. stocks slide oil slump hits energy sector; nine s&p 500's 11 sectors finish lower
u.s. stocks climb tech sector bounces back; dow, s&p 500 hit fresh records; nasdaq composi

In [ ]:
df.to_csv('./data/Combine_Label_ActualValue_normalised.csv')

In [ ]:
filename = './data/Combine_Label_ActualValue_normalised.csv'

df = pandas.read_csv(filename, encoding="UTF-8")
data = TF_Data(filename,top_words=2000)

In [ ]:
data.headlines

,normalized_headline
0,"rally legs, broad reach, --- gains s&p 500 sho..."
1,"rally legs, broad reach, --- gains s&p 500 sho..."
2,u.s. stock futures slip u.k. polls suggest con...
3,stocks fall; tech shares take brunt; tech stoc...
4,s&p 500 slips fed raises interest rates; inves...
...,...
883,"nvidia pulls s&p 500, nasdaq lower; stocks ros..."
884,dow closes higher; nvidia shares extend declin...
885,"nvidia pulls s&p 500, nasdaq lower --- stocks ..."
886,"s&p 500, nasdaq extend run gains; walgreens si..."


In [ ]:
# Predict Gite et al.'s model with preprocessed data
def test_preprocessd_data(model_name ,d):
  txt = 'preprocessed_ms_today'
  model = load_model(model_name)

  i = 0
  for text in df['normalized_headline']:
    test_sentence = data.test_sentence(text)
    result = model.predict(test_sentence)[0][0]

    if result > 0.5:
      df.loc[i,txt] = 1
    else:
      df.loc[i,txt] = 0
    df.loc[i, 'result_'+txt] = result

    i += 1

In [ ]:
# Predict Gite et al.'s model with original data
def test_origin_data(model_name ,d):
  txt = 'original_ms_today'
  model = load_model(model_name)

  i = 0
  for text in df['Title']:
    test_sentence = data.test_sentence(text)
    result = model.predict(test_sentence)[0][0]

    if result > 0.5:
      df.loc[i,txt] = 1
    else:
      df.loc[i,txt] = 0
    df.loc[i, 'result_'+txt] = result

    i += 1

In [ ]:
df

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,preprocessed_ms_today,result_preprocessed_ms_today
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.423518
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.423518
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,0.0,0.435088
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,0.0,0.428368
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,0.0,0.427217
...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",0.0,0.422160
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,0.0,0.425114
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",0.0,0.428715
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",0.0,0.420610


In [ ]:
test_preprocessd_data("./model/GiteModel_ms_today_modifiedReduceLR.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━

In [ ]:
test_origin_data("./model/GiteModel_ms_today_modifiedReduceLR.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━

In [ ]:
df

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,preprocessed_ms_today,result_preprocessed_ms_today,original_ms_today,result_original_ms_today
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.423518,0.0,0.426892
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.423518,0.0,0.426892
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,0.0,0.435088,0.0,0.436091
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,0.0,0.428368,0.0,0.430591
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,0.0,0.427217,0.0,0.428623
...,...,...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",0.0,0.422160,0.0,0.423839
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,0.0,0.425114,0.0,0.426378
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",0.0,0.428715,0.0,0.429809
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",0.0,0.420610,0.0,0.422758


In [ ]:
# Save the result of the model with modified setting of ReduceLR
df.to_csv("result_Gite_modifySettingReduceLR_ms_today_.csv")

In [ ]:
# Load new input data for the model with original setting of ReduceLR
filename = './data/Combine_Label_ActualValue_normalised.csv'

df = pandas.read_csv(filename, encoding="UTF-8")
data = TF_Data(filename,top_words=2000)

In [ ]:
# Test the sentance
test_preprocessd_data("./model/GiteModel_ms_today_originReduceLR.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━

In [ ]:
# Test the sentance
test_origin_data("./model/Model1_originalSettingReduceLR_ms_today_.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━

In [ ]:
# Save the result of the model with original setting of ReduceLR
df.to_csv("result_Model1_originalSettingReduceLR_ms_today_.csv")

In [ ]:
# Test with trained model with balanced data
test_preprocessd_data("./model/GiteModel_Augmented_ms_today_.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━

In [ ]:
df

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,preprocessed_ms_today,result_preprocessed_ms_today
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",1.0,0.538312
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",1.0,0.538312
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,1.0,0.538000
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,1.0,0.538536
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,1.0,0.538381
...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",1.0,0.538620
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,1.0,0.537420
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",1.0,0.538701
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",1.0,0.538445


In [ ]:
# Change the headline value in Tf_Data
from tf_data import TF_Data

In [ ]:
# Test with trained model with balanced data
test_origin_data("./model/GiteModel_Balanced_Augmented_ms_today_.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━

In [ ]:
# Save the result of the model with original setting of ReduceLR
df.to_csv("result_Model1_balancedData_changeTfData_ms_today_.csv")

In [ ]:
df

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,preprocessed_ms_today,result_preprocessed_ms_today,original_ms_today,result_original_ms_today
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",1.0,0.538312,1.0,0.537979
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",1.0,0.538312,1.0,0.537979
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,1.0,0.538000,1.0,0.538150
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,1.0,0.538536,1.0,0.538539
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,1.0,0.538381,1.0,0.538534
...,...,...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",1.0,0.538620,1.0,0.538312
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,1.0,0.537420,1.0,0.538050
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",1.0,0.538701,1.0,0.538668
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",1.0,0.538445,1.0,0.538401


In [ ]:
# Test with trained model with balanced data (same count)
test_preprocessd_data("./model/GiteModel_SameAugmented_ms_today_.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━

In [ ]:
# Change the headline value in Tf_Data
from tf_data import TF_Data

In [ ]:
# Test with trained model with balanced data
test_origin_data("./model/GiteModel_SameAugmented_ms_today_.keras", 'ms_today')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━

In [ ]:
df

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,preprocessed_ms_today,result_preprocessed_ms_today,original_ms_today,result_original_ms_today
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.496690,0.0,0.496638
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",0.0,0.496690,0.0,0.496638
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,0.0,0.495531,0.0,0.495824
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,0.0,0.496150,0.0,0.495941
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,0.0,0.496074,0.0,0.496032
...,...,...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",0.0,0.496071,0.0,0.495943
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,0.0,0.496714,0.0,0.496454
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",0.0,0.495797,0.0,0.495730
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",0.0,0.496635,0.0,0.496650


In [ ]:
# Save the result of the model with original setting of ReduceLR
df.to_csv("result_Model1_SamebalancedData_changeTfData_ms_today_.csv")